# REGRESSION TEST 

In [2]:
# os
import os, sys, random

# visualization
%matplotlib inline
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

# ml
import numpy as np
import torchvision as tv
import torch
from PIL import Image
from sklearn.model_selection import train_test_split

# data aug
import copy
import shutil
from collections import defaultdict
from urllib.request import urlretrieve

# helpers
import dataloader as dl
from pathlib import Path
import helpers as help

sys.path.append('../streaming_base')#
#from streaming_base.processing.processing import beamform_2d
from streaming_base.utils.utils import get_ant_pos_2d, plot_2d_polar_heatmap, plot_2d_heatmap
from utils.utility import read_radar_params

%load_ext autoreload
%autoreload 2

In [3]:
# defs
home_dir = "D:/GitHub/COM-304-Rat-Detection"
data_dir = "data/ml"

json_filename = r"scripts/chirp1"
config_lua_script = r'scripts/1843_config_streaming_task3.lua' 

In [4]:
args, chirp_dict = dl.read_lua_config(home_dir, config_lua_script)

In [5]:
names, raw_data, fft_data = help.load_data(data_dir, home_dir, json_filename, args)
virtual_ant_dict = help.make_virtual_ant_dict(fft_data, names)

In [6]:
# label data
labeled_data = {}
no_jerry_keys = ["lhs", "rhs", "left", "right"]

for name, fft in virtual_ant_dict.items():
    if (("jerry" in name.lower()) and 
        not any(key in name.lower() for key in no_jerry_keys)):
        label = 1  
    else:
        label = 0 
    labeled_data[name] = {
        "data": fft,
        "label": label
    }
    
print("Labeled data dict keys: ", labeled_data.keys())
print("Labels assigned: ")
for name in labeled_data:
    print(f"{name}: {labeled_data[name]['label']}")
    print(f"Data shape for {name}: {labeled_data[name]['data'].shape}")

Labeled data dict keys:  dict_keys(['highres_rat_setup-piped-jerry-antenna-aligned_take-3_fs-2420_slp-62-474_N-128', 'highres_rat_setup-piped-jerry_take-1_fs-2420_slp-62-474_N-128', 'highres_rat_setup-piped-jerry_take-2_fs-2420_slp-62-474_N-128', 'highres_rat_setup-pipe_take-1_fs-2420_slp-62-474_N-128', 'highres_rat_setup-pipe_take-2_fs-2420_slp-62-474_N-128', 'highres_rat_setup-pipe_take-3_fs-2420_slp-62-474_N-128', 'highres_rat_setup-pipe_take-4_fs-2420_slp-62-474_N-128', 'highres_rat_setup-pipe_take-5_fs-2420_slp-62-474_N-128', 'jerry_no-pipe_aligned1', 'jerry_no-pipe_aligned2', 'jerry_no-pipe_aligned3', 'jerry_no-pipe_aligned4', 'jerry_no-pipe_aligned5', 'jerry_no-pipe_aligned', 'jerry_no-pipe_lhs1', 'jerry_no-pipe_lhs2', 'jerry_no-pipe_lhs3', 'jerry_no-pipe_lhs4', 'jerry_no-pipe_lhs5', 'jerry_no-pipe_rhs1', 'jerry_no-pipe_rhs2', 'jerry_no-pipe_rhs3', 'jerry_no-pipe_rhs4', 'jerry_no-pipe_rhs5', 'jerry_no-pipe_wall_aligned1', 'jerry_no-pipe_wall_aligned2', 'jerry_no-pipe_wall_aligne

In [7]:
# data exploration
print(f"Total samples: {len(labeled_data)}")

label_counts = defaultdict(int)
for item in labeled_data.values():
    label_counts[item['label']] += 1
print(f"Class distribution: {label_counts}")

Total samples: 149
Class distribution: defaultdict(<class 'int'>, {1: 54, 0: 95})


In [8]:
# prep for beamform
num_rx = chirp_dict['num_rx'] 
num_tx = chirp_dict['num_tx']
num_chirps = args[2] # not sure chirp dict equiv
num_loops = chirp_dict['chirp_loops']
adc_samples = chirp_dict['samples_per_chirp']

phi_s, phi_e = 60, 130 
phi_res = 2
theta_s, theta_e = 70,110 
theta_res = 2

num_virtual_ant = num_rx * num_tx
samples_per_frame = num_virtual_ant * num_chirps * num_loops  # num_virtual_ant * samples_per_chirp * chirp_loop


theta = np.deg2rad(np.arange(theta_s, theta_e + theta_res, theta_res))
phi = np.deg2rad(np.arange(phi_s, phi_e + phi_res, phi_res))

width = 100 # azimuth width in degrees
x_locs, _, _ = get_ant_pos_2d(num_tx * num_rx, adc_samples, num_rx)
r_idxs = np.arange(100, 140, 1)

cfg_radar = {
        "range_idx": r_idxs,
        "phi": phi,
        "width": width,
        "n_radar": 1,
        "num_tx": num_tx,
        "num_rx": num_rx,
        "num_doppler": num_loops,
        "samples_per_chirp": adc_samples,
        "sample_rate": chirp_dict['sample_rate'],
        "c": 3e8,
        "lm": 3e8 / 77e9,
        "slope": chirp_dict['slope'],
        "range_res": chirp_dict['range_res'],           # NOTE : I ADDED THIS -- 4/20/2026 - KERIM
}


print(args)
print("Number of virtual antennas: ", num_virtual_ant)
print("Samples per frame: ", samples_per_frame)


[3, 4, 128, 1, '0x7', '0xF']
Number of virtual antennas:  12
Samples per frame:  1536


In [9]:
X, y = [], []
for name in labeled_data:
    data = labeled_data[name]['data']  # (frames, ants, range)
    label = labeled_data[name]['label']

    #for frame in data:
    bf_input = np.mean(data, axis=0)  # (ants, range)

    sph_pwr = help.beamform_2d(
        beat_freq_data = bf_input,
        radar_params = cfg_radar,
        x_locs = x_locs[:, 0],
    )
    bmf_out = np.abs(sph_pwr)
    img = bmf_out #np.log1p(bmf_out)   # compress dynamic range

    # normalise
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    
    X.append(img) #img
    y.append(label)

In [10]:
X = np.array(np.copy(X))       # appears ads shape (N, 180, 40)
y = np.array(np.copy(y)) 

X = X[:, None, :, :]   # (200, 1, 180, 40)
print("Final dataset shapes - X: ", X.shape, ", y: ", y.shape)
print(X.min(), X.max(), X.mean())

Final dataset shapes - X:  (149, 1, 36, 40) , y:  (149,)
0.0 1.0 0.04540339


In [11]:
# flatten for regression
X = X.reshape(X.shape[0], -1) 
print("After flattening, X shape: ", X.shape)

After flattening, X shape:  (149, 1440)


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train shapes - X: ", X_train.shape, ", y: ", y_train.shape)
print("Test shapes - X: ", X_test.shape, ", y: ", y_test.shape)

Train shapes - X:  (111, 1440) , y:  (111,)
Test shapes - X:  (38, 1440) , y:  (38,)


In [13]:
# model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.88      0.88      0.88        24
           1       0.79      0.79      0.79        14

    accuracy                           0.84        38
   macro avg       0.83      0.83      0.83        38
weighted avg       0.84      0.84      0.84        38



In [14]:
# test incase of a leak
y_train_shuffled = np.random.permutation(y_train)

model.fit(X_train, y_train_shuffled)

print(model.score(X_test, y_test))

0.6842105263157895
